# Financial Trading Signal Generation using Deep Learning Ensembles

## Import necessary libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
import plotly.graph_objects as go
import random

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import torch
from torch.utils.data import TensorDataset, DataLoader
from torch import nn

!pip install deap
import random
from deap import algorithms,base,creator,tools
!pip install optuna
import optuna


## Load Data

In [ ]:
train_path = '/content/drive/MyDrive/colab/project/data/2010-2022.csv'
test_path = '/content/drive/MyDrive/colab/project/data/2023.csv'

In [ ]:
train_df = pd.read_csv(train_path, parse_dates=['Date'])
test_df = pd.read_csv(test_path, parse_dates=['Date'])
print(train_df.info())
print(train_df.head())

In [ ]:
# convert Date col to datetime
train_df['Date'] = pd.to_datetime(train_df['Date'])

## Exploratory Data Analysis


#### Closing prices vs Time

In [ ]:
plt.figure(figsize=(12,6))
plt.plot(train_df['Date'], train_df['Close'], label='Closing Price', color='blue')
plt.xlabel('Date')
plt.ylabel('Stock Price')
plt.title('Google Stock Closing Price Over Time')
plt.legend()
plt.show()

#### Trading vol vs Time


In [ ]:
plt.figure(figsize=(12,6))
plt.plot(train_df['Date'], train_df['Volume'], label='Trading Volume', color='orange')
plt.xlabel('Date')
plt.ylabel('Volume')
plt.title('Google Stock Trading Volume Over Time')
plt.legend()
plt.show()

## Feature Engineering

#### Simple Moving Average (50 days) and Exponential Moving Average (50 days)

In [ ]:
# Simple Moving Average (SMA) - 50-day window
train_df['SMA_50'] = train_df['Close'].rolling(window=50).mean()

# Exponential Moving Average (EMA) - 50-day window
train_df['EMA_50'] = train_df['Close'].ewm(span=50, adjust=False).mean()

plt.figure(figsize=(12,6))
plt.plot(train_df['Date'], train_df['Close'], label='Closing Price', color='blue', alpha=0.5)
plt.plot(train_df['Date'], train_df['SMA_50'], label='SMA 50', color='red')
plt.plot(train_df['Date'], train_df['EMA_50'], label='EMA 50', color='green')
plt.xlabel('Date')
plt.ylabel('Stock Price')
plt.title('Moving Averages (SMA & EMA)')
plt.legend()
plt.show()


In [ ]:
def compute_rsi(data, window=14):
    delta = data['Close'].diff(1)
    gain = (delta.where(delta > 0, 0)).rolling(window=window).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=window).mean()
    rs = gain / loss
    return 100 - (100 / (1 + rs))

train_df['RSI_14'] = compute_rsi(train_df, window=14)

# Plot RSI
plt.figure(figsize=(12,6))
plt.plot(train_df['Date'], train_df['RSI_14'], label='RSI 14-day', color='purple')
plt.axhline(70, linestyle='--', color='red', label='Overbought (70)')
plt.axhline(30, linestyle='--', color='green', label='Oversold (30)')
plt.xlabel('Date')
plt.ylabel('RSI Value')
plt.title('Relative Strength Index (RSI)')
plt.legend()
plt.show()


In [ ]:
def compute_bollinger_bands(data, window=20):
    sma = data['Close'].rolling(window=window).mean()
    std = data['Close'].rolling(window=window).std()

    upper_band = sma + (2 * std)
    lower_band = sma - (2 * std)

    return sma, upper_band, lower_band

# Compute Bollinger Bands
train_df['SMA_20'], train_df['Upper_BB'], train_df['Lower_BB'] = compute_bollinger_bands(train_df)

# Plot Bollinger Bands
plt.figure(figsize=(12,6))
plt.plot(train_df['Date'], train_df['Close'], label='Closing Price', color='blue', alpha=0.5)
plt.plot(train_df['Date'], train_df['SMA_20'], label='SMA 20', color='orange')
plt.plot(train_df['Date'], train_df['Upper_BB'], label='Upper Bollinger Band', color='red', linestyle='dashed')
plt.plot(train_df['Date'], train_df['Lower_BB'], label='Lower Bollinger Band', color='green', linestyle='dashed')
plt.xlabel('Date')
plt.ylabel('Stock Price')
plt.title('Bollinger Bands')
plt.legend()
plt.show()


#### Feature Correlations



In [ ]:
plt.figure(figsize=(10,6))
sns.heatmap(train_df.corr(), annot=True, cmap='coolwarm', fmt=".2f")
plt.title("Feature Correlation Matrix")
plt.show()


In [ ]:
def compute_rsi(data, window=14):
    delta = data['Close'].diff()
    gain = delta.where(delta > 0, 0).rolling(window=window).mean()
    loss = -delta.where(delta < 0, 0).rolling(window=window).mean()
    rs = gain / loss
    return 100 - (100 / (1 + rs))

def compute_bollinger_bands(data, window=20):
    sma = data['Close'].rolling(window).mean()
    std = data['Close'].rolling(window).std()
    return sma, sma + 2*std, sma - 2*std

In [ ]:
for df in [train_df, test_df]:
    df['SMA_50'] = df['Close'].rolling(window=50).mean()
    df['EMA_50'] = df['Close'].ewm(span=50, adjust=False).mean()
    df['RSI_14'] = compute_rsi(df, window=14)
    df['SMA_20'], df['Upper_BB'], df['Lower_BB'] = compute_bollinger_bands(df)

In [ ]:
train_df.dropna(inplace=True)
test_df.dropna(inplace=True)

In [ ]:
train_df.info()

## Training

#### Normalize Features


Scaled input features perform better when scaled. Stock price values which have large numerical values are to be brought between 0 and 1 using sklearn.MinMaxScaler

In [ ]:
features = ['Close', 'SMA_50', 'EMA_50', 'RSI_14', 'SMA_20', 'Upper_BB', 'Lower_BB']
scaler = MinMaxScaler()
train_scaled = scaler.fit_transform(train_df[features].values)

#### Create Sequences for Time-Series Modeling
Recurrent neural networks (RNNs), LSTMs, and GRUs require sequential data as input. We will create rolling windowed sequences where each input sample consists of lookback days of past data to predict the next day's price.

In [ ]:
sequence_length = 60

X_train, y_train = [], []
for i in range(sequence_length, len(train_scaled)):
    X_train.append(train_scaled[i-sequence_length:i])
    y_train.append(train_scaled[i,0]) # we need Close price as target

X_train_tensor = torch.tensor(np.array(X_train), dtype=torch.float32)
y_train_tensor = torch.tensor(np.array(y_train), dtype=torch.float32)

train_loader = DataLoader(TensorDataset(X_train_tensor, y_train_tensor), batch_size=32, shuffle=True)

#### Train-Test Split
We will split our dataset into training (80%) and testing (20%) sets to evaluate model performance.

In [ ]:
combined_df = pd.concat((train_df, test_df), axis=0)
combined_scaled = scaler.transform(combined_df[features].values)
test_scaled = combined_scaled[-(len(test_df) + sequence_length):]

X_test, y_test = [], []
for i in range(sequence_length, len(test_scaled)):
    X_test.append(test_scaled[i-sequence_length:i])
    y_test.append(test_scaled[i, 0])  # Close price

X_test_tensor = torch.tensor(np.array(X_test), dtype=torch.float32)
y_test_tensor = torch.tensor(np.array(y_test), dtype=torch.float32)
test_loader = DataLoader(TensorDataset(X_test_tensor, y_test_tensor), batch_size=32, shuffle=False)


### Baseline RNN Model

Before building more complex models like LSTM and GRU, we will first implement a simple Recurrent Neural Network (RNN) as a baseline. This will help us establish a benchmark for future improvements.

In [ ]:
class RNNModel(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        self.rnn1 = nn.RNN(input_size, 300, batch_first=True)
        self.rnn2 = nn.RNN(300, 100, batch_first=True)
        self.rnn3 = nn.RNN(100, 100, batch_first=True)
        self.rnn4 = nn.RNN(100, 100, batch_first=True)
        self.dropout = nn.Dropout(0.2)
        self.fc = nn.Linear(100, 1)

    def forward(self, x):
        out, _ = self.rnn1(x)
        out = self.dropout(out)
        out, _ = self.rnn2(out)
        out = self.dropout(out)
        out, _ = self.rnn3(out)
        out = self.dropout(out)
        out, _ = self.rnn4(out)
        out = self.dropout(out)
        return self.fc(out[:, -1, :])

In [ ]:
rnn_model = RNNModel(input_size=len(features))
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(rnn_model.parameters(), lr=0.001)
loss_history_rnn = []
epochs = 20
for epoch in range(epochs):
    rnn_model.train()
    epoch_loss = 0.0
    for X_batch, y_batch in train_loader:
        output = rnn_model(X_batch)
        loss = criterion(output.squeeze(), y_batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    loss_history_rnn.append(epoch_loss)
    if (epoch+1) % 4 == 0:
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.6f}")

In [ ]:
rnn_model.eval()
with torch.no_grad():
    predictions = rnn_model(X_test_tensor).numpy()

pad_shape = (len(predictions), len(features))
full_pred = np.zeros(pad_shape)
full_pred[:, 0] = predictions.flatten()
predictions_inverse = scaler.inverse_transform(full_pred)[:, 0]


result_df = test_df[['Date']].copy().iloc[-len(predictions_inverse):]
result_df['RNN_Predicted_Close'] = predictions_inverse
result_df['Actual_Close'] = test_df['Close'].values[-len(predictions_inverse):]

fig = go.Figure()
fig.add_trace(go.Scatter(x=result_df['Date'], y=result_df['Actual_Close'],
                         mode='lines', name='Actual Price', line=dict(color='blue')))
fig.add_trace(go.Scatter(x=result_df['Date'], y=result_df['RNN_Predicted_Close'],
                         mode='lines', name='Predicted Price', line=dict(color='red')))
fig.update_layout(title='RNN Prediction',
                  xaxis_title='Date', yaxis_title='Price',
                  hovermode='x unified', template='plotly_white')
fig.show()

In [ ]:
torch.save(rnn_model.state_dict(), "/content/drive/MyDrive/colab/project/models/rnn_model.pth")

In [ ]:
class LSTMModel(nn.Module):
    def __init__(self, input_size=len(features), hidden_dim1=300, hidden_dim2=100, hidden_dim3=100, hidden_dim4=100, dropout=0.2):
        super(LSTMModel, self).__init__()
        self.lstm1 = nn.LSTM(input_size=input_size, hidden_size =hidden_dim1, batch_first=True, dropout=dropout, num_layers=1)
        self.lstm2 = nn.LSTM(input_size=hidden_dim1 , hidden_size=hidden_dim2, batch_first= True, dropout=dropout, num_layers=1)
        self.lstm3 = nn.LSTM(input_size=hidden_dim2 , hidden_size= hidden_dim3 , batch_first=True, dropout=dropout, num_layers=1)
        self.lstm4 = nn.LSTM(input_size=hidden_dim3, hidden_size=hidden_dim4, batch_first= True, dropout=dropout,   num_layers=1)

        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim4, 1)

    def forward(self, x):
        out, _ = self.lstm1(x)
        out = self.dropout(out)
        out, _ = self.lstm2(out)
        out = self.dropout(out)
        out, _ = self.lstm3(out)
        out = self.dropout(out)
        out, _ = self.lstm4(out)
        out = self.dropout(out)
        return self.fc(out[:, -1, :])

In [ ]:
lstm_model = LSTMModel(input_size=len(features))
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(lstm_model.parameters(), lr=0.001)

In [ ]:
epochs = 20
batch_size = 32
loss_history_lstm = []
for epoch in range(epochs):
    permutation = torch.randperm(X_train_tensor.size(0))
    epoch_loss = 0.0

    for i in range(0, X_train_tensor.size(0), batch_size):
        indices = permutation[i:i+batch_size]
        batch_X, batch_y = X_train_tensor[indices], y_train_tensor[indices]

        optimizer.zero_grad()
        outputs = lstm_model(batch_X).squeeze()
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    loss_history_lstm.append(epoch_loss)
    if (epoch+1) % 4 == 0:
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.6f}")

In [ ]:
epochs = 20
batch_size = 32
loss_history_lstm = []
for epoch in range(epochs):
    permutation = torch.randperm(X_train_tensor.size(0))
    epoch_loss = 0.0

    for i in range(0, X_train_tensor.size(0), batch_size):
        indices = permutation[i:i+batch_size]
        batch_X, batch_y = X_train_tensor[indices], y_train_tensor[indices]

        optimizer.zero_grad()
        outputs = lstm_model(batch_X).squeeze()
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    loss_history_lstm.append(epoch_loss)
    if (epoch+1) % 4 == 0:
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.6f}")

In [ ]:
lstm_model.eval()
with torch.no_grad():
    lstm_predictions = lstm_model(X_test_tensor).cpu().numpy()

pad_shape = (len(lstm_predictions), len(features))
full_pred = np.zeros(pad_shape)
full_pred[:, 0] = lstm_predictions.flatten()
lstm_predictions_inverse = scaler.inverse_transform(full_pred)[:, 0]

result_df['LSTM_Predicted_Close'] = lstm_predictions_inverse

fig = go.Figure()
fig.add_trace(go.Scatter(x=result_df['Date'], y=result_df['Actual_Close'],
                         mode='lines', name='Actual Price', line=dict(color='blue')))
fig.add_trace(go.Scatter(x=result_df['Date'], y=result_df['RNN_Predicted_Close'],
                         mode='lines', name='RNN Predicted', line=dict(color='red')))
fig.add_trace(go.Scatter(x=result_df['Date'], y=result_df['LSTM_Predicted_Close'],
                         mode='lines', name='LSTM Predicted', line=dict(color='green')))
fig.update_layout(title='Stock Price Prediction: RNN vs LSTM',
                  xaxis_title='Date', yaxis_title='Stock Price',
                  template='plotly_white', hovermode='x unified')
fig.show()

In [ ]:
torch.save(lstm_model.state_dict(), "/content/drive/MyDrive/colab/project/models/lstm_model.pth")

In [ ]:
class GRUModel(nn.Module):
    def __init__(self, input_size=len(features), hidden_dim1=300, hidden_dim2=100, hidden_dim3=100, hidden_dim4=100, dropout=0.2):
        super(GRUModel, self).__init__()

        self.gru1 = nn.GRU(input_size=input_size, hidden_size=hidden_dim1, batch_first=True, dropout=dropout, num_layers=1)
        self.gru2 = nn.GRU(input_size=hidden_dim1, hidden_size=hidden_dim2, batch_first=True, dropout=dropout, num_layers=1)
        self.gru3 = nn.GRU(input_size=hidden_dim2, hidden_size=hidden_dim3, batch_first=True, dropout=dropout, num_layers=1)
        self.gru4 = nn.GRU(input_size=hidden_dim3, hidden_size=hidden_dim4, batch_first=True, dropout=dropout, num_layers=1)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim4, 1)

    def forward(self, x):
        out, _ = self.gru1(x)
        out = self.dropout(out)
        out, _ = self.gru2(out)
        out = self.dropout(out)
        out, _ = self.gru3(out)
        out = self.dropout(out)
        out, _ = self.gru4(out)
        out = self.dropout(out)
        return self.fc(out[:, -1, :])

In [ ]:
gru_model = GRUModel(input_size=len(features))
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(gru_model.parameters(), lr=0.001)

In [ ]:
# get losses
def train_model(model, optimizer, train_loader, test_data, epochs=20, model_name="Model"):
    train_losses = []
    test_losses = []
    X_test, y_test = test_data

    for epoch in range(epochs):
        model.train()
        epoch_train_loss = 0.0

        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()
            output = model(X_batch).squeeze()
            loss = criterion(output, y_batch)
            loss.backward()
            optimizer.step()
            epoch_train_loss += loss.item()

        train_losses.append(epoch_train_loss / len(train_loader))

        # evaluate
        model.eval()
        with torch.no_grad():
            predictions = model(X_test).squeeze()
            test_loss = mean_squared_error(y_test.cpu(), predictions.cpu())
            test_losses.append(test_loss)

        if (epoch + 1) % 4 == 0:
            print(f"{model_name} Epoch [{epoch+1}/{epochs}] Train Loss: {train_losses[-1]:.6f}, Test Loss: {test_loss:.6f}")

    return train_losses, test_losses


In [ ]:

rnn_model = RNNModel(input_size=len(features))
optimizer_rnn = torch.optim.Adam(rnn_model.parameters(), lr=0.001)
train_rnn, test_rnn = train_model(rnn_model, optimizer_rnn, train_loader, (X_test_tensor, y_test_tensor), model_name="RNN")

lstm_model = LSTMModel(input_size=len(features))
optimizer_lstm = torch.optim.Adam(lstm_model.parameters(), lr=0.001)
train_lstm, test_lstm = train_model(lstm_model, optimizer_lstm, train_loader, (X_test_tensor, y_test_tensor), model_name="LSTM")

gru_model = GRUModel(input_size=len(features))
optimizer_gru = torch.optim.Adam(gru_model.parameters(), lr=0.001)
train_gru, test_gru = train_model(gru_model, optimizer_gru, train_loader, (X_test_tensor, y_test_tensor), model_name="GRU")

In [ ]:
def plot_losses(train, test, title):
    plt.plot(train, label='Train Loss')
    plt.plot(test, label='Test Loss')
    plt.title(f'{title} Loss Over Epochs')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)
    plt.show()

plot_losses(train_rnn, test_rnn, "RNN")
plot_losses(train_lstm, test_lstm, "LSTM")
plot_losses(train_gru, test_gru, "GRU")


In [ ]:
import pickle

# Save all losses
loss_data = {
    'RNN': {'train': train_rnn, 'test': test_rnn},
    'LSTM': {'train': train_lstm, 'test': test_lstm},
    'GRU': {'train': train_gru, 'test': test_gru},
}

with open('loss_histories.pkl', 'wb') as f:
    pickle.dump(loss_data, f)

print("Loss histories saved to 'loss_histories.pkl'")

In [ ]:
epochs = 20
batch_size = 32
loss_history_gru = []
for epoch in range(epochs):
    permutation = torch.randperm(X_train_tensor.size(0))
    epoch_loss = 0.0

    for i in range(0, X_train_tensor.size(0), batch_size):
        indices = permutation[i:i+batch_size]
        batch_X, batch_y = X_train_tensor[indices], y_train_tensor[indices]
        optimizer.zero_grad()
        outputs = gru_model(batch_X).squeeze()
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    loss_history_gru.append(epoch_loss)
    if (epoch+1) % 4 == 0:
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.6f}")

In [ ]:
plt.figure(figsize=(10, 6))

# plt.plot(loss_history_rnn, label='RNN Loss')
plt.plot(loss_history_lstm, label='LSTM Loss')
# plt.plot(loss_history_gru, label='GRU Loss')

plt.title("Training Loss per Epoch")
plt.xlabel("Epoch")
plt.ylabel("Loss (MSE)")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
gru_model.eval()
with torch.no_grad():
    gru_predictions = gru_model(X_test_tensor).cpu().numpy()


pad_shape = (len(gru_predictions), len(features))
full_pred = np.zeros(pad_shape)
full_pred[:, 0] = gru_predictions.flatten()
gru_predictions_inverse = scaler.inverse_transform(full_pred)[:, 0]

result_df['GRU_Predicted_Close'] = gru_predictions_inverse

fig = go.Figure()
fig.add_trace(go.Scatter(x=result_df['Date'], y=result_df['Actual_Close'],
                         mode='lines', name='Actual Price', line=dict(color='blue')))
fig.add_trace(go.Scatter(x=result_df['Date'], y=result_df['RNN_Predicted_Close'],
                         mode='lines', name='RNN Predicted', line=dict(color='red')))
fig.add_trace(go.Scatter(x=result_df['Date'], y=result_df['LSTM_Predicted_Close'],
                         mode='lines', name='LSTM Predicted', line=dict(color='green')))
fig.add_trace(go.Scatter(x=result_df['Date'], y=result_df['GRU_Predicted_Close'],
                         mode='lines', name='GRU Predicted', line=dict(color='orange')))

fig.update_layout(title='Stock Price Prediction: RNN vs LSTM vs GRU',
                  xaxis_title='Date',
                  yaxis_title='Stock Price',
                  template='plotly_white',
                  hovermode='x unified')
fig.show()

In [ ]:
torch.save(gru_model.state_dict(), "/content/drive/MyDrive/colab/project/models/gru_model.pth")

In [ ]:
result_df.to_csv('/content/drive/MyDrive/colab/project/result_df.csv')

In [ ]:
rnn_model = RNNModel(input_size=len(features))
lstm_model = LSTMModel(input_size=len(features))
gru_model = GRUModel(input_size=len(features))

rnn_model.load_state_dict(torch.load("/content/drive/MyDrive/colab/project/models/rnn_model.pth"))
lstm_model.load_state_dict(torch.load("/content/drive/MyDrive/colab/project/models/lstm_model.pth"))
gru_model.load_state_dict(torch.load("/content/drive/MyDrive/colab/project/models/gru_model.pth"))

rnn_model.eval()
lstm_model.eval()
gru_model.eval()

result_df = pd.read_csv('/content/drive/MyDrive/colab/project/result_df.csv')

In [ ]:
def evaluate_model(true, pred):
    mae = mean_absolute_error(true, pred)
    mse = mean_squared_error(true, pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(true, pred)
    return mae, mse, rmse, r2

actual = result_df['Actual_Close'].values
rnn_pred = result_df['RNN_Predicted_Close'].values
lstm_pred = result_df['LSTM_Predicted_Close'].values
gru_pred = result_df['GRU_Predicted_Close'].values

models = {'RNN': rnn_pred, 'LSTM': lstm_pred, 'GRU': gru_pred}
metrics = {}

for name, pred in models.items():
    metrics[name] = evaluate_model(actual, pred)
for rnn_model, (mae, mse, rmse, r2) in metrics.items():
    print(f"{rnn_model}: MAE: {mae:.4f}, MSE: {mse:.4f}, RMSE: {rmse:.4f}, R2score: {r2:.4f}")


## Hyperparameter tuning the best model

#### Deap

In [ ]:
random.seed(42)
torch.manual_seed(42)

def evaluate_gru(individual):
    hidden_dim1, hidden_dim2, hidden_dim3, hidden_dim4, dropout, lr = individual
    dropout = float(dropout)
    lr = float(lr)

    model = GRUModel(input_size=len(features),
                     hidden_dim1=hidden_dim1,
                     hidden_dim2=hidden_dim2,
                     hidden_dim3=hidden_dim3,
                     hidden_dim4=hidden_dim4,
                     dropout=dropout)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()

    epochs = 5
    batch_size = 32
    model.train()

    for epoch in range(epochs):
        permutation = torch.randperm(X_train_tensor.size(0))
        for i in range(0, X_train_tensor.size(0), batch_size):
            indices = permutation[i:i+batch_size]
            batch_X, batch_y = X_train_tensor[indices], y_train_tensor[indices]
            optimizer.zero_grad()
            outputs = model(batch_X).squeeze()
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()

    model.eval()
    with torch.no_grad():
        preds = model(X_test_tensor).squeeze()
        rmse = torch.sqrt(nn.MSELoss()(preds, y_test_tensor)).item()
    return (rmse,)

creator.create("FitnessMin", base.Fitness, weights=(-1.0,))
creator.create("Individual", list, fitness=creator.FitnessMin)

toolbox = base.Toolbox()
toolbox.register("hidden_dim1", random.randint, 64, 512)
toolbox.register("hidden_dim2", random.randint, 32, 256)
toolbox.register("hidden_dim3", random.randint, 32, 256)
toolbox.register("hidden_dim4", random.randint, 32, 256)
toolbox.register("dropout", random.uniform, 0.1, 0.5)
toolbox.register("lr", random.uniform, 1e-4, 5e-3)

toolbox.register("individual", tools.initCycle, creator.Individual,
                 (toolbox.hidden_dim1, toolbox.hidden_dim2, toolbox.hidden_dim3, toolbox.hidden_dim4, toolbox.dropout, toolbox.lr),
                 n=1)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

toolbox.register("evaluate", evaluate_gru)
toolbox.register("mate", tools.cxBlend, alpha=0.5)
toolbox.register("mutate", tools.mutGaussian, mu=0, sigma=20, indpb=0.2)
toolbox.register("select", tools.selTournament, tournsize=3)

def run_deap_gru_tuning(pop_size=10, generations=5):
    pop = toolbox.population(n=pop_size)
    hof = tools.HallOfFame(1)

    stats = tools.Statistics(lambda ind: ind.fitness.values[0])
    stats.register("avg", lambda x: sum(x)/len(x))
    stats.register("min", min)
    pop, logbook = algorithms.eaSimple(pop, toolbox, cxpb=0.5, mutpb=0.3,
                                       ngen=generations, stats=stats,
                                       halloffame=hof, verbose=True)

    print("Best hyperparameters:", hof[0])
    print("Best RMSE:", hof[0].fitness.values[0])
    return hof[0]

In [ ]:
deap_gru_params = run_deap_gru_tuning(pop_size=10, generations=5)
print(deap_gru_params)

 #### Optuna


In [ ]:
def te(model, optimizer, criterion, train_loader, val_loader, epochs=100):
    for epoch in range(epochs):
        model.train()
        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()
            output = model(X_batch).squeeze()
            loss = criterion(output, y_batch)
            loss.backward()
            optimizer.step()

    model.eval()
    val_losses = []
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            output = model(X_batch).squeeze()
            loss = criterion(output, y_batch)
            val_losses.append(loss.item())
    return sum(val_losses) / len(val_losses)

In [ ]:
def gru_objective(trial):
    hidden1 = trial.suggest_int('hidden1', 128, 512)
    hidden2 = trial.suggest_int('hidden2', 64, 256)
    hidden3 = trial.suggest_int('hidden3', 64, 256)
    hidden4 = trial.suggest_int('hidden4', 64, 256)
    dropout = trial.suggest_float('dropout', 0.1, 0.5)
    lr = trial.suggest_float('lr', 1e-4, 1e-2, log=True)

    class TunedGRU(nn.Module):
        def __init__(self):
            super().__init__()
            self.gru1 = nn.GRU(input_size=len(features), hidden_size=hidden1, batch_first=True, dropout=dropout)
            self.gru2 = nn.GRU(input_size=hidden1, hidden_size=hidden2, batch_first=True, dropout=dropout)
            self.gru3 = nn.GRU(input_size=hidden2, hidden_size=hidden3, batch_first=True, dropout=dropout)
            self.gru4 = nn.GRU(input_size=hidden3, hidden_size=hidden4, batch_first=True, dropout=dropout)
            self.dropout = nn.Dropout(dropout)
            self.fc = nn.Linear(hidden4, 1)

        def forward(self, x):
            out, _ = self.gru1(x)
            out = self.dropout(out)
            out, _ = self.gru2(out)
            out = self.dropout(out)
            out, _ = self.gru3(out)
            out = self.dropout(out)
            out, _ = self.gru4(out)
            out = self.dropout(out)
            return self.fc(out[:, -1, :])

    model = TunedGRU()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()

    return te(model, optimizer, criterion, train_loader, test_loader, epochs=10)


In [ ]:
study_gru = optuna.create_study(direction='minimize')
study_gru.optimize(gru_objective, n_trials=20)
print('Best GRU parameters: ', study_gru.best_params)

In [ ]:
# Best GRU parameters:  {'hidden1': 187, 'hidden2': 174, 'hidden3': 189, 'hidden4': 198, 'dropout': 0.26563665610213577, 'lr': 0.0018600287493190672}
tuned_gru_model = GRUModel(
    input_size=len(features),
    hidden_dim1=187,
    hidden_dim2=174,
    hidden_dim3=189,
    hidden_dim4=198,
    dropout=0.2656
)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(tuned_gru_model.parameters(), lr=0.00186)

epochs = 50
batch_size = 32
loss_history_gru = []
for epoch in range(epochs):
    permutation = torch.randperm(X_train_tensor.size(0))
    epoch_loss = 0.0

    for i in range(0, X_train_tensor.size(0), batch_size):
        indices = permutation[i:i+batch_size]
        batch_X, batch_y = X_train_tensor[indices], y_train_tensor[indices]
        optimizer.zero_grad()
        outputs = gru_model(batch_X).squeeze()
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    loss_history_gru.append(epoch_loss)
    if (epoch+1) % 10 == 0:
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.6f}")

In [ ]:
tuned_gru_model.eval()
with torch.no_grad():
    gru_predictions = tuned_gru_model(X_test_tensor).cpu().numpy()

pad_shape = (len(gru_predictions), len(features))
full_pred = np.zeros(pad_shape)
full_pred[:, 0] = gru_predictions.flatten()
tuned_gru_predictions_inverse = scaler.inverse_transform(full_pred)[:, 0]

# result_df['Tuned_GRU_Predicted_Close'] = tuned_gru_predictions_inverse

fig = go.Figure()
fig.add_trace(go.Scatter(x=result_df['Date'], y=result_df['Actual_Close'],
                         mode='lines', name='Actual Price', line=dict(color='blue')))
fig.add_trace(go.Scatter(x=result_df['Date'], y=result_df['RNN_Predicted_Close'],
                         mode='lines', name='RNN Predicted', line=dict(color='red')))
fig.add_trace(go.Scatter(x=result_df['Date'], y=result_df['LSTM_Predicted_Close'],
                         mode='lines', name='LSTM Predicted', line=dict(color='green')))
fig.add_trace(go.Scatter(x=result_df['Date'], y=result_df['GRU_Predicted_Close'],
                         mode='lines', name='Tuned GRU Predicted', line=dict(color='orange')))

fig.update_layout(title='Stock Price Prediction: RNN vs LSTM vs Tuned GRU',
                  xaxis_title='Date',
                  yaxis_title='Stock Price',
                  template='plotly_white',
                  hovermode='x unified')
fig.show()

In [ ]:
torch.save(tuned_gru_model.state_dict(), "/content/drive/MyDrive/colab/project/models/tuned_gru_model.pth")

## Ensemble our preds


In [ ]:
result_df['Ensemble_Predicted_Close'] = (
    result_df['RNN_Predicted_Close'] +
    result_df['LSTM_Predicted_Close'] +
    result_df['GRU_Predicted_Close']
) / 3

In [ ]:
result_df.to_csv('/content/drive/MyDrive/colab/project/result_df.csv')

In [ ]:
result_df = pd.read_csv('/content/drive/MyDrive/colab/project/result_df.csv')

### Trading Signals

In [ ]:
def direction_accuracy(true, pred):
    true_diff = np.diff(true)
    pred_diff = np.diff(pred)
    correct = np.sum((true_diff > 0) == (pred_diff > 0))
    return correct / len(true_diff)

for name, pred in models.items():
    acc = direction_accuracy(actual, pred)
    print(f"{name} direction accuracy : {acc*100:.2f}% ")


In [ ]:
fig = go.Figure()

fig.add_trace(go.Scatter(x=result_df['Date'], y=result_df['Actual_Close'], name='Actual', line=dict(color='blue')))
fig.add_trace(go.Scatter(x=result_df['Date'], y=result_df['RNN_Predicted_Close'], name='RNN', line=dict(color='red')))
fig.add_trace(go.Scatter(x=result_df['Date'], y=result_df['LSTM_Predicted_Close'], name='LSTM', line=dict(color='green')))
fig.add_trace(go.Scatter(x=result_df['Date'], y=result_df['GRU_Predicted_Close'], name='GRU', line=dict(color='orange')))
fig.add_trace(go.Scatter(x=result_df['Date'], y=result_df['Ensemble_Predicted_Close'], name='Ensemble', line=dict(color='blue', dash='dot')))

fig.update_layout(title='Predictions vs Actual',
                  xaxis_title='Date',
                  yaxis_title='Stock Price',
                  template='plotly_white')
fig.show()


In [ ]:
def get_trade_signal(current, predicted, threshold=0.1):
    pct_change = (predicted - current) / current
    if pct_change > threshold:
        return 'Buy'
    elif pct_change < -threshold:
        return 'Sell'
    return 'Hold'

for rnn_model in ['RNN_Predicted_Close', 'LSTM_Predicted_Close', 'GRU_Predicted_Close']:
    signal_col = rnn_model.replace('Close', 'Signal')
    result_df[signal_col] = result_df.apply(lambda row: get_trade_signal(row['Actual_Close'], row[rnn_model]), axis=1)


In [ ]:
result_df['Ensemble_Predicted_Signal'] = result_df.apply(
    lambda row: get_trade_signal(row['Actual_Close'], row['Ensemble_Predicted_Close']), axis=1
)

In [ ]:
result_df.head()